# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{}'
)
""".format(HF_TOKEN))

rel = "hf://datasets/FlyRank/internship-warehouse"

In [26]:
feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [27]:
feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice and why

I chose a Random Forest Classifier because the task is a binary classification problem (predicting whether impressions will increase the next day). Random Forest can capture non-linear relationships between features and is more robust than a single Decision Tree. It also provides feature importance, making the model easier to interpret.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

I used an 80/20 train-test split with stratification to preserve the class distribution in both training and testing datasets. This provides a fair evaluation because both sets contain similar proportions of positive and negative examples.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Model vs Baseline

| Model | Accuracy | Precision | Recall | F1 Score |
|-------|---------:|----------:|--------:|---------:|
| Random Forest | 0.756 | 0.424 | 0.825 | 0.561 |

The Random Forest achieved good recall, meaning it successfully identified most of the pages whose impressions increased. Precision is lower, indicating that some predicted increases did not actually occur. This trade-off is acceptable when the goal is to detect as many improving pages as possible.

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=30,
                       n_jobs=-1, random_state=42)

In [29]:
y_pred = rf.predict(X_test)

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.756492282552633
Precision: 0.4244523392573161
Recall   : 0.82537181679014
F1 Score : 0.5606084611159231

Classification Report

              precision    recall  f1-score   support

           0       0.95      0.74      0.83   1544017
           1       0.42      0.83      0.56    357972

    accuracy                           0.76   1901989
   macro avg       0.69      0.78      0.70   1901989
weighted avg       0.85      0.76      0.78   1901989



In [31]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
0,gsc_impressions,0.679299
2,avg_position,0.300579
3,ctr,0.019901
1,gsc_clicks,0.000221


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Feature Importance

The most important feature was **gsc_impressions** (67.9%), followed by **avg_position** (30.1%). CTR contributed only a small amount, while gsc_clicks had almost no impact on the model. This suggests that current impressions and search position are the strongest indicators of future impression growth.

## Error Analysis

The model produced high recall but relatively low precision. This means it correctly identified most pages with future growth but also predicted growth for some pages that did not actually improve. These false positives may occur because impression changes depend on factors that are not included in the available features, such as seasonal trends or external search behavior.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.